In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
from datetime import datetime, timezone

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F


spark = SparkSession.builder.getOrCreate()


BRONZE_PATH = (
    "/Volumes/big_data_project/project/data/"
    "spotify_dataset/charts_songs_daily.parquet"
)

NEW_SILVER_BASE_PATH = (
    "/Volumes/big_data_project/project/data/"
    "silver_revised/charts_songs_daily"
)

STAGING_PATH = f"{NEW_SILVER_BASE_PATH}/_staging"
SILVER_UNPARTITIONED_PATH = f"{NEW_SILVER_BASE_PATH}/unpartitioned"
SILVER_PARTITIONED_PATH = (
    f"{NEW_SILVER_BASE_PATH}/partitioned_by_year_month"
)
INVALID_QUARANTINE_PATH = (
    f"{NEW_SILVER_BASE_PATH}/quarantine/invalid_records"
)
DUPLICATE_QUARANTINE_PATH = (
    f"{NEW_SILVER_BASE_PATH}/quarantine/business_key_duplicates"
)

PIPELINE_RUN_ID = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

BUSINESS_KEY = ["date", "country", "uri"]

ALLOWED_ENTRY_STATUS = [
    "MOVED_UP",
    "MOVED_DOWN",
    "NO_CHANGE",
    "NEW_ENTRY",
    "RE_ENTRY",
]

SOURCE_COLUMNS = [
    "date",
    "country",
    "rank",
    "uri",
    "artist_names",
    "track_name",
    "label",
    "peak_rank",
    "previous_rank",
    "days_on_chart",
    "streams",
    "consecutive_days",
    "entry_status",
    "peak_date",
    "entry_rank",
    "entry_date",
    "release_date",
    "artist_uris",
]

MANDATORY_COLUMNS = [
    "date",
    "country",
    "rank",
    "uri",
    "peak_rank",
    "previous_rank",
    "days_on_chart",
    "streams",
    "consecutive_days",
    "entry_status",
    "peak_date",
    "entry_rank",
    "entry_date",
    "artist_uris",
]


def path_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


def assert_required_columns(df, required_columns, dataframe_name):
    missing_columns = sorted(set(required_columns) - set(df.columns))
    if missing_columns:
        raise ValueError(
            f"{dataframe_name} is missing columns: {missing_columns}"
        )


def clean_display_string(column_name):
    cleaned_value = F.regexp_replace(
        F.trim(F.col(column_name).cast("string")),
        r"\s+",
        " ",
    )
    return F.when(cleaned_value == "", F.lit(None)).otherwise(cleaned_value)


def normalized_text(column_expression):
    normalized_value = F.lower(
        F.regexp_replace(
            F.trim(column_expression.cast("string")),
            r"\s+",
            " ",
        )
    )
    normalized_value = F.regexp_replace(
        normalized_value,
        r"[\.,;]+$",
        "",
    )
    return F.when(normalized_value == "", F.lit(None)).otherwise(
        normalized_value
    )


def count_duplicate_key_groups(df):
    return (
        df.groupBy(*BUSINESS_KEY)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )


def count_mandatory_null_rows(df):
    null_condition = None
    for column_name in MANDATORY_COLUMNS:
        current_condition = F.col(column_name).isNull()
        null_condition = (
            current_condition
            if null_condition is None
            else null_condition | current_condition
        )
    return df.filter(null_condition).count()


def write_parquet(df, path, partition_columns=None):
    writer = (
        df.write
        .format("parquet")
        .mode("overwrite")
        .option("overwriteSchema", "true")
    )
    if partition_columns:
        writer = writer.partitionBy(*partition_columns)
    writer.save(path)


def read_bronze_data():
    if not path_exists(BRONZE_PATH):
        raise FileNotFoundError(f"Bronze source not found: {BRONZE_PATH}")

    bronze_df = spark.read.parquet(BRONZE_PATH)
    assert_required_columns(bronze_df, SOURCE_COLUMNS, "Bronze dataframe")
    return bronze_df.select(*SOURCE_COLUMNS)


def enforce_schema(bronze_df):
    return bronze_df.select(
        F.expr("try_cast(date as date)").alias("date"),
        clean_display_string("country").alias("country"),
        F.expr("try_cast(rank as int)").alias("rank"),
        clean_display_string("uri").alias("uri"),
        clean_display_string("artist_names").alias("artist_names"),
        clean_display_string("track_name").alias("track_name"),
        clean_display_string("label").alias("label"),
        F.expr("try_cast(peak_rank as int)").alias("peak_rank"),
        F.expr("try_cast(previous_rank as int)").alias("previous_rank"),
        F.expr("try_cast(days_on_chart as int)").alias("days_on_chart"),
        F.expr("try_cast(streams as bigint)").alias("streams"),
        F.expr("try_cast(consecutive_days as int)").alias(
            "consecutive_days"
        ),
        clean_display_string("entry_status").alias("entry_status"),
        F.expr("try_cast(peak_date as date)").alias("peak_date"),
        F.expr("try_cast(entry_rank as int)").alias("entry_rank"),
        F.expr("try_cast(entry_date as date)").alias("entry_date"),
        F.expr("try_cast(release_date as date)").alias("release_date"),
        clean_display_string("artist_uris").alias("artist_uris"),
    )


def standardize_text_values(df):
    return (
        df.withColumn("country", F.lower(F.col("country")))
        .withColumn("entry_status", F.upper(F.col("entry_status")))
        .withColumn(
            "artist_names",
            F.coalesce(F.col("artist_names"), F.lit("Unknown Artist")),
        )
        .withColumn(
            "track_name",
            F.coalesce(F.col("track_name"), F.lit("Unknown Track")),
        )
        .withColumn(
            "label",
            F.coalesce(F.col("label"), F.lit("Unknown Label")),
        )
    )


def add_artist_collaboration_columns(df):
    artist_name_array = F.transform(
        F.split(F.col("artist_names"), r"\s*,\s*"),
        lambda item: normalized_text(item),
    )
    artist_name_array = F.filter(
        artist_name_array,
        lambda item: item.isNotNull() & (item != ""),
    )

    artist_uri_array = F.transform(
        F.split(F.col("artist_uris"), r"\|"),
        lambda item: F.trim(item),
    )
    artist_uri_array = F.filter(
        artist_uri_array,
        lambda item: item.isNotNull() & (item != ""),
    )

    return (
        df.withColumn("_artist_name_array", artist_name_array)
        .withColumn("_artist_uri_array", artist_uri_array)
        .withColumn(
            "primary_artist_name_normalized",
            F.when(
                F.col("artist_names") == "Unknown Artist",
                F.lit("unknown artist"),
            ).otherwise(F.element_at(F.col("_artist_name_array"), 1)),
        )
        .withColumn(
            "collaboration_artist_names_normalized",
            F.when(
                F.size(F.col("_artist_name_array")) > 1,
                F.array_join(
                    F.slice(
                        F.col("_artist_name_array"),
                        2,
                        F.size(F.col("_artist_name_array")) - 1,
                    ),
                    ", ",
                ),
            ).otherwise(F.lit("No Collaboration")),
        )
        .withColumn(
            "primary_artist_uri",
            F.element_at(F.col("_artist_uri_array"), 1),
        )
        .withColumn(
            "collaboration_artist_uris",
            F.when(
                F.size(F.col("_artist_uri_array")) > 1,
                F.array_join(
                    F.slice(
                        F.col("_artist_uri_array"),
                        2,
                        F.size(F.col("_artist_uri_array")) - 1,
                    ),
                    "|",
                ),
            ).otherwise(F.lit("No Collaboration URI")),
        )
        .withColumn(
            "artist_count",
            F.greatest(
                F.size(F.col("_artist_name_array")),
                F.size(F.col("_artist_uri_array")),
            ),
        )
        .withColumn("is_collaboration", F.col("artist_count") > 1)
        .drop("_artist_name_array", "_artist_uri_array")
    )


def add_normalized_grouping_columns(df):
    return (
        df.withColumn(
            "track_name_normalized",
            normalized_text(F.col("track_name")),
        )
        .withColumn(
            "label_normalized",
            normalized_text(F.col("label")),
        )
    )


def add_validation_reasons(df):
    mandatory_null_condition = None
    for column_name in MANDATORY_COLUMNS:
        current_condition = F.col(column_name).isNull()
        mandatory_null_condition = (
            current_condition
            if mandatory_null_condition is None
            else mandatory_null_condition | current_condition
        )

    validation_reasons = F.array(
        F.when(mandatory_null_condition, F.lit("MANDATORY_NULL")),
        F.when(
            ~(
                (F.col("country") == "global")
                | F.col("country").rlike(r"^[a-z]{2}$")
            ),
            F.lit("INVALID_COUNTRY"),
        ),
        F.when(~F.col("rank").between(1, 200), F.lit("INVALID_RANK")),
        F.when(
            ~F.col("peak_rank").between(1, 200),
            F.lit("INVALID_PEAK_RANK"),
        ),
        F.when(
            ~(
                (F.col("previous_rank") == -1)
                | F.col("previous_rank").between(1, 200)
            ),
            F.lit("INVALID_PREVIOUS_RANK"),
        ),
        F.when(
            ~F.col("entry_rank").between(1, 200),
            F.lit("INVALID_ENTRY_RANK"),
        ),
        F.when(F.col("streams") <= 0, F.lit("INVALID_STREAMS")),
        F.when(
            F.col("days_on_chart") < 1,
            F.lit("INVALID_DAYS_ON_CHART"),
        ),
        F.when(
            ~(
                (F.col("consecutive_days") >= 1)
                & (F.col("consecutive_days") <= F.col("days_on_chart"))
            ),
            F.lit("INVALID_CONSECUTIVE_DAYS"),
        ),
        F.when(
            ~F.col("entry_status").isin(ALLOWED_ENTRY_STATUS),
            F.lit("INVALID_ENTRY_STATUS"),
        ),
        F.when(
            F.col("entry_date") > F.col("date"),
            F.lit("ENTRY_DATE_AFTER_CHART_DATE"),
        ),
        F.when(
            ~F.col("peak_date").between(
                F.col("entry_date"),
                F.col("date"),
            ),
            F.lit("INVALID_PEAK_DATE"),
        ),
        F.when(
            F.col("release_date").isNotNull()
            & (F.col("release_date") > F.col("date")),
            F.lit("RELEASE_DATE_AFTER_CHART_DATE"),
        ),
    )

    return df.withColumn(
        "_validation_reasons",
        F.filter(
            validation_reasons,
            lambda reason: reason.isNotNull(),
        ),
    )


def split_valid_and_invalid(df):
    invalid_df = (
        df.filter(F.size(F.col("_validation_reasons")) > 0)
        .withColumn(
            "validation_reason",
            F.concat_ws("|", F.col("_validation_reasons")),
        )
        .withColumn("pipeline_run_id", F.lit(PIPELINE_RUN_ID))
        .withColumn("quarantined_at", F.current_timestamp())
        .drop("_validation_reasons")
    )

    valid_df = (
        df.filter(F.size(F.col("_validation_reasons")) == 0)
        .drop("_validation_reasons")
        .withColumn(
            "release_date_quality_flag",
            F.when(
                F.col("release_date").isNull(),
                F.lit("MISSING_RELEASE_DATE"),
            ).otherwise(F.lit("VALID")),
        )
    )

    return valid_df, invalid_df


def deduplicate_business_keys(valid_df):
    hash_columns = [
        F.coalesce(F.col(column_name).cast("string"), F.lit("NULL"))
        for column_name in valid_df.columns
    ]

    ranked_df = (
        valid_df.withColumn(
            "_row_hash",
            F.sha2(F.concat_ws("||", *hash_columns), 256),
        )
        .withColumn(
            "_duplicate_row_number",
            F.row_number().over(
                Window.partitionBy(*BUSINESS_KEY).orderBy(
                    F.col("streams").desc_nulls_last(),
                    F.col("days_on_chart").desc_nulls_last(),
                    F.col("consecutive_days").desc_nulls_last(),
                    F.col("_row_hash").asc(),
                )
            ),
        )
        .withColumn(
            "_duplicate_group_size",
            F.count(F.lit(1)).over(Window.partitionBy(*BUSINESS_KEY)),
        )
    )

    deduplicated_df = (
        ranked_df.filter(F.col("_duplicate_row_number") == 1)
        .drop(
            "_row_hash",
            "_duplicate_row_number",
            "_duplicate_group_size",
        )
    )

    duplicate_df = (
        ranked_df.filter(F.col("_duplicate_row_number") > 1)
        .withColumn(
            "duplicate_reason",
            F.lit("DUPLICATE_BUSINESS_KEY"),
        )
        .withColumn("pipeline_run_id", F.lit(PIPELINE_RUN_ID))
        .withColumn("quarantined_at", F.current_timestamp())
        .drop("_row_hash")
    )

    return deduplicated_df, duplicate_df


def engineer_dashboard_features(df):
    return (
        df.withColumn("chart_year", F.year("date"))
        .withColumn("chart_month", F.month("date"))
        .withColumn(
            "days_since_entry",
            F.datediff(F.col("date"), F.col("entry_date")),
        )
        .withColumn(
            "days_since_release",
            F.when(
                F.col("release_date").isNotNull(),
                F.datediff(F.col("date"), F.col("release_date")),
            ).otherwise(F.lit(None).cast("int")),
        )
        .withColumn(
            "daily_rank_change",
            F.when(
                F.col("previous_rank").between(1, 200),
                F.col("previous_rank") - F.col("rank"),
            ).otherwise(F.lit(None).cast("int")),
        )
        .withColumn(
            "rank_movement",
            F.when(
                ~F.col("previous_rank").between(1, 200),
                F.lit("NO_PREVIOUS_RANK"),
            )
            .when(
                F.col("previous_rank") - F.col("rank") > 0,
                F.lit("MOVED_UP"),
            )
            .when(
                F.col("previous_rank") - F.col("rank") < 0,
                F.lit("MOVED_DOWN"),
            )
            .otherwise(F.lit("NO_CHANGE")),
        )
        .withColumn("pipeline_run_id", F.lit(PIPELINE_RUN_ID))
        .withColumn("silver_created_at", F.current_timestamp())
    )


FINAL_SILVER_COLUMNS = [
    "date",
    "country",
    "rank",
    "uri",
    "artist_names",
    "track_name",
    "label",
    "peak_rank",
    "previous_rank",
    "days_on_chart",
    "streams",
    "consecutive_days",
    "entry_status",
    "peak_date",
    "entry_rank",
    "entry_date",
    "release_date",
    "artist_uris",
    "primary_artist_name_normalized",
    "collaboration_artist_names_normalized",
    "primary_artist_uri",
    "collaboration_artist_uris",
    "artist_count",
    "is_collaboration",
    "track_name_normalized",
    "label_normalized",
    "release_date_quality_flag",
    "chart_year",
    "chart_month",
    "days_since_entry",
    "days_since_release",
    "daily_rank_change",
    "rank_movement",
    "pipeline_run_id",
    "silver_created_at",
]


def validate_staging(staging_df, expected_row_count):
    assert_required_columns(
        staging_df,
        FINAL_SILVER_COLUMNS,
        "Staging dataframe",
    )

    actual_row_count = staging_df.count()
    duplicate_groups = count_duplicate_key_groups(staging_df)
    mandatory_null_rows = count_mandatory_null_rows(staging_df)

    if actual_row_count != expected_row_count:
        raise RuntimeError(
            "Staging row count mismatch. "
            f"Expected {expected_row_count}, found {actual_row_count}."
        )

    if duplicate_groups != 0:
        raise RuntimeError("Duplicate business keys remain in staging.")

    if mandatory_null_rows != 0:
        raise RuntimeError("Mandatory NULL values remain in staging.")

    print("Staging validation passed.")


def validate_final_output(source_rows, invalid_rows, duplicate_rows):
    unpartitioned_df = spark.read.parquet(SILVER_UNPARTITIONED_PATH)
    partitioned_df = spark.read.parquet(SILVER_PARTITIONED_PATH)

    unpartitioned_rows = unpartitioned_df.count()
    partitioned_rows = partitioned_df.count()
    duplicate_groups = count_duplicate_key_groups(unpartitioned_df)
    mandatory_null_rows = count_mandatory_null_rows(unpartitioned_df)

    reconciled_rows = unpartitioned_rows + invalid_rows + duplicate_rows

    if unpartitioned_rows != partitioned_rows:
        raise RuntimeError(
            "Partitioned and unpartitioned row counts do not match."
        )

    if reconciled_rows != source_rows:
        raise RuntimeError("Source-to-output row reconciliation failed.")

    if duplicate_groups != 0:
        raise RuntimeError(
            "Final Silver contains duplicate business keys."
        )

    if mandatory_null_rows != 0:
        raise RuntimeError(
            "Final Silver contains mandatory NULL values."
        )

    print()
    print("REVISED BRONZE TO SILVER PIPELINE COMPLETED")
    print("===========================================")
    print(f"Source rows: {source_rows}")
    print(f"Invalid rows quarantined: {invalid_rows}")
    print(f"Duplicate rows excluded: {duplicate_rows}")
    print(f"Final Silver rows: {unpartitioned_rows}")
    print(f"Final Silver columns: {len(unpartitioned_df.columns)}")
    print(f"Duplicate business-key groups: {duplicate_groups}")
    print(f"Mandatory NULL rows: {mandatory_null_rows}")
    print(f"Pipeline run ID: {PIPELINE_RUN_ID}")
    print(f"Unpartitioned output: {SILVER_UNPARTITIONED_PATH}")
    print(f"Partitioned output: {SILVER_PARTITIONED_PATH}")
    print("Bronze source was not modified.")


def main():
    print("STARTING REVISED SPOTIFY SONG SILVER PIPELINE")
    print("=============================================")
    print(f"Pipeline run ID: {PIPELINE_RUN_ID}")

    bronze_df = read_bronze_data()
    source_rows = bronze_df.count()

    print(f"Source rows: {source_rows}")
    print(f"Source columns: {len(bronze_df.columns)}")

    typed_df = enforce_schema(bronze_df)
    standardized_df = standardize_text_values(typed_df)
    artist_enriched_df = add_artist_collaboration_columns(
        standardized_df
    )
    normalized_df = add_normalized_grouping_columns(
        artist_enriched_df
    )

    validated_df = add_validation_reasons(normalized_df)
    valid_df, invalid_df = split_valid_and_invalid(validated_df)

    invalid_rows = invalid_df.count()
    write_parquet(invalid_df, INVALID_QUARANTINE_PATH)
    print(f"Invalid rows quarantined: {invalid_rows}")

    deduplicated_df, duplicate_df = deduplicate_business_keys(valid_df)

    duplicate_rows = duplicate_df.count()
    write_parquet(duplicate_df, DUPLICATE_QUARANTINE_PATH)
    print(f"Duplicate rows excluded: {duplicate_rows}")

    silver_df = (
        engineer_dashboard_features(deduplicated_df)
        .select(*FINAL_SILVER_COLUMNS)
    )

    expected_silver_rows = (
        source_rows - invalid_rows - duplicate_rows
    )

    write_parquet(silver_df, STAGING_PATH)
    staging_df = spark.read.parquet(STAGING_PATH)
    validate_staging(staging_df, expected_silver_rows)

    write_parquet(
        staging_df,
        SILVER_UNPARTITIONED_PATH,
    )

    write_parquet(
        staging_df,
        SILVER_PARTITIONED_PATH,
        partition_columns=["chart_year", "chart_month"],
    )

    validate_final_output(
        source_rows,
        invalid_rows,
        duplicate_rows,
    )

    try:
        dbutils.fs.rm(STAGING_PATH, recurse=True)
        print("Temporary staging output removed.")
    except Exception as error:
        print(
            "Final outputs are valid, but staging cleanup failed: "
            f"{error}"
        )

    print()
    print("Read the unpartitioned Silver dataset:")
    print(
        "silver_df = spark.read.parquet("
        f"'{SILVER_UNPARTITIONED_PATH}'"
        ")"
    )

    print()
    print("Read the partitioned Silver dataset:")
    print(
        "silver_partitioned_df = spark.read.parquet("
        f"'{SILVER_PARTITIONED_PATH}'"
        ")"
    )


if __name__ == "__main__":
    main()

STARTING REVISED SPOTIFY SONG SILVER PIPELINE
Pipeline run ID: 20260722T125417_563065Z
Source rows: 42757018
Source columns: 18
Invalid rows quarantined: 2659828
Duplicate rows excluded: 192
Staging validation passed.

REVISED BRONZE TO SILVER PIPELINE COMPLETED
Source rows: 42757018
Invalid rows quarantined: 2659828
Duplicate rows excluded: 192
Final Silver rows: 40096998
Final Silver columns: 35
Duplicate business-key groups: 0
Mandatory NULL rows: 0
Pipeline run ID: 20260722T125417_563065Z
Unpartitioned output: /Volumes/big_data_project/project/data/silver_revised/charts_songs_daily/unpartitioned
Partitioned output: /Volumes/big_data_project/project/data/silver_revised/charts_songs_daily/partitioned_by_year_month
Bronze source was not modified.
Temporary staging output removed.

Read the unpartitioned Silver dataset:
silver_df = spark.read.parquet('/Volumes/big_data_project/project/data/silver_revised/charts_songs_daily/unpartitioned')

Read the partitioned Silver dataset:
silver_pa